# 03 — Merge multi-fuente (código inline)
**Etapa:** 2b — Multi-corpus EN→ES

Une `data/interim/<fuente>/data.parquet` con sus traducciones (caché
JSONL append-only de MarianMT o `data_es.parquet` si la traducción terminó)
y produce `data/processed/corpus_v1.parquet` con todas las fuentes + columnas
de metadatos de traducción (`is_translated`, `mt_system`, `text_orig`, `lang_orig`).

Funciona con traducciones **parciales** (usa el JSONL si `data_es.parquet`
todavía no se escribió). Re-ejecutar el notebook periódicamente para ir
incorporando docs a medida que avanza la traducción.

**Toda la lógica de merge está inline** — no usa `src.data.merge_corpora`.

**Pre-requisito**: `02_translate_multi.ipynb` corriendo en background
(cachea JSONL append-only por corpus).

**Output**: `data/processed/corpus_v1.parquet` con `coello_guilarte` +
los corpus traducidos disponibles al momento del merge.

In [ ]:
# --- Imports y setup ---
import os, sys, json, time
from pathlib import Path
import pandas as pd
import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

INTERIM = Path("data/interim")
PROCESSED = Path("data/processed")
DATA_CFG = yaml.safe_load(Path("configs/data.yaml").read_text(encoding="utf-8"))
SCHEMA = ["doc_id", "user_id", "source", "timestamp", "text_raw", "text_clean",
          "label", "label_source", "lang"]
EXTRA = ["text_orig", "lang_orig", "is_translated", "mt_system"]

# Estado actual de traducciones (cuántas hay en JSONL por corpus)
for src in ["kaggle_sdd", "reddit_mh_posts", "depression_reddit"]:
    tr_dir = INTERIM / src / "translations"
    parq = INTERIM / src / "data_es.parquet"
    n_jsonl = 0
    if tr_dir.exists():
        for j in tr_dir.glob("*.jsonl"):
            n_jsonl += sum(1 for _ in open(j, encoding="utf-8"))
    n_parq = "(no escrito)"
    if parq.exists():
        n_parq = f"{len(pd.read_parquet(parq)):,}"
    print(f"{src}: {n_jsonl:,} en JSONL | {n_parq} en data_es.parquet")
print(f"\nRepo: {REPO_ROOT}")

In [ ]:
# --- Merge inline (sin src/) ---
def attach_translation(df: pd.DataFrame, translated: pd.DataFrame) -> pd.DataFrame:
    """Reemplaza text_clean por su traducción, conservando el original."""
    tr = translated[["doc_id", "text_es", "mt_system"]]
    out = df.merge(tr, on="doc_id", how="inner")
    out["text_orig"] = out["text_clean"]
    out["lang_orig"] = out["lang"]
    out["text_clean"] = out["text_es"]
    out["lang"] = "es"
    out["is_translated"] = True
    return out.drop(columns=["text_es"])


def load_translations(src_dir: Path) -> pd.DataFrame | None:
    """Lee data_es.parquet si existe; si no, fallback al JSONL append-only."""
    p_es = src_dir / "data_es.parquet"
    if p_es.exists():
        return pd.read_parquet(p_es)
    tr_dir = src_dir / "translations"
    if not tr_dir.exists():
        return None
    p_jsonl = next(tr_dir.glob("*.jsonl"), None)
    if p_jsonl is None:
        return None
    print(f"  (usando cache parcial: {p_jsonl.name})")
    rows = []
    for line in p_jsonl.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    tr_df = pd.DataFrame(rows).rename(columns={"translation": "text_es"})
    if "mt_system" not in tr_df.columns:
        tr_df["mt_system"] = ""
    return tr_df[["doc_id", "text_es", "mt_system"]]


frames: list[pd.DataFrame] = []
for src, meta in DATA_CFG["data"]["sources"].items():
    if not meta.get("enabled", False) or not meta.get("merge", True):
        continue
    p = INTERIM / src / "data.parquet"
    if not p.exists():
        print(f"  (saltando {src}: no existe {p})")
        continue
    df = pd.read_parquet(p)
    for col in SCHEMA:
        if col not in df.columns:
            df[col] = "" if col != "label" else 0
    df = df[SCHEMA].copy()
    if meta.get("translate", False):
        translated = load_translations(INTERIM / src)
        if translated is None or len(translated) == 0:
            print(f"  (saltando {src}: sin traducciones)")
            continue
        n_before = len(df)
        df = attach_translation(df, translated)
        print(f"  {src}: {len(df):,}/{n_before:,} con match de traducción")
    else:
        df["text_orig"] = df["text_clean"]
        df["lang_orig"] = df["lang"]
        df["is_translated"] = False
        df["mt_system"] = ""
    df = df[SCHEMA + EXTRA].copy()
    df["corpus_source"] = src
    frames.append(df)
    print(f"  -> {src}: {len(df):,} filas cargadas")

if not frames:
    raise SystemExit("No hay interim/<fuente>/data.parquet para mergear.")

corpus = pd.concat(frames, ignore_index=True)
before = len(corpus)
corpus = corpus.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)
print(f"\ndedup: {before:,} → {len(corpus):,}")

PROCESSED.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED / "corpus_v1.parquet"
corpus.to_parquet(out_path, index=False)
print(f"-> {out_path}: {len(corpus):,} filas")

In [ ]:
# --- Verificar schema + stats ---
df = pd.read_parquet("data/processed/corpus_v1.parquet")
print(f"Filas: {len(df):,}")
print(f"\nColumnas ({len(df.columns)}): {df.columns.tolist()}")
print(f"\nPor corpus_source:\n{df['corpus_source'].value_counts()}")
print(f"\nPor lang:\n{df['lang'].value_counts()}")
print(f"\nis_translated:\n{df['is_translated'].value_counts()}")
print(f"\nlabel:\n{df['label'].value_counts()}")
print(f"\nmt_system (top 3):\n{df['mt_system'].value_counts().head(3)}")

n_src = df["corpus_source"].nunique()
assert n_src >= 2, f"esperaba >=2 fuentes, hay {n_src}"
assert (df["lang"] == "es").all(), "toda la columna lang debería ser 'es'"
n_translated = int(df["is_translated"].sum())
print(f"\n-> {n_translated:,} traducciones (subirá a >200k cuando termine kaggle + reddit_mh + depression_reddit)")
print("OK — schema y stats consistentes (parcial OK; final cuando terminen las 3 traducciones)")

# Guardar stats JSON
stats = {
    "n_rows": int(len(df)),
    "n_users": int(df["user_id"].nunique()),
    "by_source": df["source"].value_counts().to_dict(),
    "by_label": df["label"].value_counts().to_dict(),
    "n_translated": n_translated,
}
(PROCESSED / "corpus_statistics.json").write_text(
    json.dumps(stats, indent=2, ensure_ascii=False)
)
print("-> corpus_statistics.json")

In [ ]:
# --- QC: una muestra de textos traducidos vs originales ---
import random
random.seed(42)
translated = df[df["is_translated"]]
if len(translated) >= 3:
    sample = translated.sample(3, random_state=42)
    for _, row in sample.iterrows():
        print(f"--- {row['corpus_source']} | label={row['label']} ---")
        orig = str(row['text_orig'])[:200]
        es = str(row['text_clean'])[:200]
        print(f"  ORIG (en): {orig}...")
        print(f"  ES:        {es}...")
        print()
else:
    print(f"(solo {len(translated)} traducciones, salteo muestra)")

## Conclusiones

- `data/processed/corpus_v1.parquet` se actualiza en cada corrida del merge.
- `coello_guilarte` siempre presente (~1M, ES nativo).
- Corpus traducidos se agregan a medida que el JSONL crece (kaggle_sdd primero, después reddit_mh_posts y depression_reddit).
- Columnas de trazabilidad: `is_translated`, `mt_system`, `text_orig`, `lang_orig`.
- **Siguiente**: `04_build_splits_multi.ipynb` para splits user-level multi.